# 03 — Silver: Clima Atual (Current Weather)
## Módulo 2A | Camada Silver | Seção do Exame: S3 (22%)

**O que este notebook faz:**
- Lê a Delta Table Bronze `raw_current_weather`
- Parseia o JSON com `from_json()` usando `StructType` explícito
- Aplica tipagem estrita: timestamps Unix → `TimestampType`, coordenadas → `DoubleType`
- Implementa regras de Data Quality com flag `_dq_status` (sem descartar registros)
- Realiza `MERGE INTO` na Silver por chave `city_id + weather_timestamp` (idempotente)

**Princípio Silver:** Schema fixo, tipos corretos, deduplicação por chave de negócio.  
Registros ruins **não são descartados** — são flaggeados para auditoria.

---

### ✅ Checklist deste notebook
- [ ] Célula 1 — StructType definido sem erros de sintaxe
- [ ] Célula 2 — `from_json` parseia corretamente: `df.printSchema()` mostra tipos corretos
- [ ] Célula 3 — Campos extraídos: `df.select("city_id","city_name","temp","weather_timestamp").show()`
- [ ] Célula 4 — `_dq_status` gerado: `df.groupBy("_dq_status").count().show()`
- [ ] Célula 5 — Tabela `silver.current_weather` criada
- [ ] Célula 6 — MERGE INTO executado sem erro
- [ ] Re-executei e count da Silver **não dobrou** (idempotência confirmada)

---
## Célula 1 — Schema Explícito do JSON (StructType)

**Por que schema explícito em vez de `inferSchema`?**
- Em produção, schema inference é custoso (lê todos os dados antes de processar)
- Schema explícito garante que novos campos inesperados da API **não quebram** o pipeline
- Erros de tipo ficam explícitos e rastreáveis

**Estrutura do JSON retornado pelo `/weather`:**
```json
{
  "id": 3448439, "name": "São Paulo",
  "coord": {"lat": -23.5, "lon": -46.6},
  "main": {"temp": 22.5, "feels_like": 22.1, "humidity": 65, "pressure": 1013},
  "weather": [{"main": "Clear", "description": "clear sky"}],
  "wind": {"speed": 3.5},
  "sys": {"country": "BR"},
  "dt": 1704067200
}
```

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, LongType, DoubleType, FloatType,
    StringType, ArrayType
)
from delta.tables import DeltaTable

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
dbutils.widgets.text("catalog", "workshop_weather", "Catálogo UC")
CATALOG = dbutils.widgets.get("catalog")

BRONZE_TABLE = f"{CATALOG}.bronze.raw_current_weather"
SILVER_TABLE = f"{CATALOG}.silver.current_weather"

# Schema do JSON retornado pela OpenWeatherMap /weather
# Campos não declarados aqui são ignorados — sem quebra de pipeline
WEATHER_SCHEMA = StructType([
    StructField("id",   IntegerType(), True),
    StructField("name", StringType(),  True),
    StructField("coord", StructType([
        StructField("lat", DoubleType(), True),
        StructField("lon", DoubleType(), True),
    ]), True),
    StructField("main", StructType([
        StructField("temp",       FloatType(),   True),
        StructField("feels_like", FloatType(),   True),
        StructField("humidity",   IntegerType(), True),
        StructField("pressure",   IntegerType(), True),
    ]), True),
    StructField("weather", ArrayType(StructType([
        StructField("main",        StringType(), True),
        StructField("description", StringType(), True),
    ])), True),
    StructField("wind", StructType([
        StructField("speed", FloatType(), True),
    ]), True),
    StructField("sys", StructType([
        StructField("country", StringType(), True),
    ]), True),
    StructField("dt", LongType(), True),  # Unix epoch timestamp
])

print("✅ Schema definido com sucesso.")
print(f"   Bronze : {BRONZE_TABLE}")
print(f"   Silver : {SILVER_TABLE}")

---
## Célula 2 — Leitura da Bronze + Parsing do JSON

`from_json()` converte a string `raw_payload` em colunas tipadas usando o schema definido acima.

In [0]:
# Lê apenas registros com sucesso na Bronze
df_bronze = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("_status") == "success")
    .filter(F.col("raw_payload").isNotNull())
)

# Parseia o JSON usando o schema explícito
# Campos ausentes no JSON ficam como null — sem erro
df_parsed = df_bronze.withColumn(
    "data",
    F.from_json(F.col("raw_payload"), WEATHER_SCHEMA)
)

print("Schema do DataFrame após parsing:")
df_parsed.select("data").printSchema()
print(f"Registros lidos da Bronze: {df_parsed.count()}")

---
## Célula 3 — Extração e Tipagem dos Campos

**Conversões aplicadas:**
- `dt` (Unix epoch / Long) → `weather_timestamp` (TimestampType) via `from_unixtime`
- `coord.lat` / `coord.lon` → `DoubleType` (já tipado no schema)
- `weather[0].main` → `weather_main` (primeiro elemento do array)

In [0]:
df_silver_staging = df_parsed.select(
    # Identificação
    F.col("data.id")                              .alias("city_id"),
    F.col("data.name")                            .alias("city_name"),
    F.col("data.sys.country")                     .alias("country"),
    F.col("data.coord.lat")                       .alias("lat"),
    F.col("data.coord.lon")                       .alias("lon"),
    # Timestamp: Unix epoch → TimestampType
    F.from_unixtime(F.col("data.dt"))             .cast("timestamp").alias("weather_timestamp"),
    # Condições meteorológicas
    F.col("data.main.temp")                       .alias("temp"),
    F.col("data.main.feels_like")                 .alias("feels_like"),
    F.col("data.main.humidity")                   .alias("humidity"),
    F.col("data.main.pressure")                   .alias("pressure"),
    F.col("data.weather")[0]["main"]              .alias("weather_main"),
    F.col("data.weather")[0]["description"]       .alias("weather_description"),
    F.col("data.wind.speed")                      .alias("wind_speed"),
    # Metadados de controle
    F.col("_batch_id"),
    F.col("_ingestion_timestamp"),
    F.current_timestamp()                         .alias("_processed_timestamp"),
)

print("Amostra dos dados extraídos:")
df_silver_staging.select(
    "city_id", "city_name", "country", "temp", "weather_timestamp"
).show(truncate=False)

---
## Célula 4 — Data Quality: flag `_dq_status`

**Regras de DQ implementadas:**
- `city_id` não pode ser nulo
- `weather_timestamp` não pode ser nula
- `temp` deve estar entre -80°C e 60°C (range fisicamente plausível)

**Por que não descartar registros inválidos?**  
Descartar silenciosamente esconde problemas de upstream. O flag `_dq_status = 'invalid'`  
permite auditoria, alertas e reprocessamento futuro sem perder dados.

In [0]:
df_with_dq = df_silver_staging.withColumn(
    "_dq_status",
    F.when(
        F.col("city_id").isNull() |
        F.col("weather_timestamp").isNull() |
        F.col("temp").isNull() |
        (F.col("temp") < -80) |
        (F.col("temp") > 60),
        F.lit("invalid")
    ).otherwise(F.lit("valid"))
)

print("Distribuição de _dq_status:")
df_with_dq.groupBy("_dq_status").count().show()

# Mostra registros inválidos se existirem (para diagnóstico)
invalid = df_with_dq.filter(F.col("_dq_status") == "invalid")
if invalid.count() > 0:
    print("⚠️  Registros inválidos encontrados:")
    invalid.select("city_name", "temp", "weather_timestamp", "_dq_status").show()
else:
    print("✅ Todos os registros passaram nas regras de DQ.")

---
## Célula 5 — Criar a Tabela Silver (se não existir)

Criamos a tabela com DDL explícito para garantir o schema correto independente dos dados.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workshop_weather.silver.current_weather (
    city_id               INT           COMMENT 'ID único da cidade na OpenWeatherMap',
    city_name             STRING        COMMENT 'Nome da cidade',
    country               STRING        COMMENT 'Código do país (ex: BR)',
    lat                   DOUBLE        COMMENT 'Latitude',
    lon                   DOUBLE        COMMENT 'Longitude',
    weather_timestamp     TIMESTAMP     COMMENT 'Timestamp da medição (convertido de Unix epoch)',
    temp                  FLOAT         COMMENT 'Temperatura em Celsius',
    feels_like            FLOAT         COMMENT 'Sensação térmica em Celsius',
    humidity              INT           COMMENT 'Umidade relativa (%)',
    pressure              INT           COMMENT 'Pressão atmosférica (hPa)',
    weather_main          STRING        COMMENT 'Condição principal (Clear, Rain, Clouds...)',
    weather_description   STRING        COMMENT 'Descrição detalhada da condição',
    wind_speed            FLOAT         COMMENT 'Velocidade do vento (m/s)',
    _batch_id             STRING        COMMENT 'ID do batch de ingestão Bronze',
    _ingestion_timestamp  TIMESTAMP     COMMENT 'Timestamp de ingestão na Bronze',
    _processed_timestamp  TIMESTAMP     COMMENT 'Timestamp de processamento Silver',
    _dq_status            STRING        COMMENT 'Resultado da validação de qualidade: valid | invalid'
)
USING DELTA
COMMENT 'Tabela Silver: clima atual por cidade, com DQ aplicado e upsert por city_id + weather_timestamp'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'  -- habilita CDF para uso no Módulo 4 e Desafio 1
);

---
## Célula 6 — MERGE INTO (Upsert)

**Por que MERGE e não overwrite ou append?**
- `append` → duplicaria dados a cada re-execução
- `overwrite` → perderia histórico
- `MERGE INTO` → **idempotente por chave de negócio**: se o registro já existe, atualiza; se é novo, insere

**Chave de negócio:** `city_id + weather_timestamp`  
Uma cidade pode ter apenas **um** registro por timestamp de medição.

In [0]:
# Registra o DataFrame de staging como view temporária
# para ser referenciado no MERGE INTO SQL
df_with_dq.createOrReplaceTempView("staging_current_weather")

spark.sql(f"""
MERGE INTO {SILVER_TABLE} AS target
USING staging_current_weather AS source
ON  target.city_id           = source.city_id
AND target.weather_timestamp = source.weather_timestamp

-- Registro já existe: atualiza todos os campos (dados podem ter sido corrigidos upstream)
WHEN MATCHED THEN UPDATE SET
    target.city_name             = source.city_name,
    target.country               = source.country,
    target.lat                   = source.lat,
    target.lon                   = source.lon,
    target.temp                  = source.temp,
    target.feels_like            = source.feels_like,
    target.humidity              = source.humidity,
    target.pressure              = source.pressure,
    target.weather_main          = source.weather_main,
    target.weather_description   = source.weather_description,
    target.wind_speed            = source.wind_speed,
    target._batch_id             = source._batch_id,
    target._ingestion_timestamp  = source._ingestion_timestamp,
    target._processed_timestamp  = source._processed_timestamp,
    target._dq_status            = source._dq_status

-- Registro novo: insere
WHEN NOT MATCHED THEN INSERT *
""")

print("✅ MERGE INTO executado com sucesso.")

---
## Célula 7 — Validação da Tabela Silver

In [0]:
%sql
-- Contagem por cidade — deve ter exatamente 1 linha por cidade (última medição)
-- Re-executar o notebook NÃO deve aumentar este count
SELECT
    city_name,
    country,
    ROUND(temp, 1)      AS temp_C,
    humidity            AS humidity_pct,
    weather_main,
    weather_timestamp,
    _dq_status
FROM workshop_weather.silver.current_weather
ORDER BY city_name;

In [0]:
%sql
-- Confirma que o MERGE é idempotente:
-- COUNT deve permanecer igual após re-execuções do notebook
SELECT COUNT(*) AS total_registros_silver
FROM workshop_weather.silver.current_weather;

---
## ✅ Checklist Final — Módulo 2A concluído?

- [ ] `printSchema()` mostra `weather_timestamp` como `timestamp` (não Long)
- [ ] `lat` e `lon` são `double` (não string)
- [ ] `_dq_status` presente em todos os registros
- [ ] Silver tem 4 linhas (uma por cidade) após múltiplas re-execuções
- [ ] MERGE não duplicou — idempotência confirmada
- [ ] CDF habilitado: `delta.enableChangeDataFeed = true` (usado no Módulo 4 e Desafio 1)

**Próximo: `04_process_silver_forecast.ipynb`**

---
*Workshop: Engenharia de Dados com Databricks & Delta Lake — OpenWeatherMap API*